# Raw data audit

Module 05 ? Mosamiat

The CLI validates raw JSON and backfill hashes before generating these artifacts. Run the command in docs/eda.md first. This notebook reads the published audit; it makes no network requests.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from mosamiat.backfill import digest

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/mosamiat").is_dir()
)
OUTPUT = project_root / "artifacts/eda-smoke"  # Change for another published EDA run.
manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
assert manifest["status"] == "complete", "Run mosamiat-eda first"
assert all(digest(OUTPUT / name) == checksum for name, checksum in manifest["sha256"].items())
summary = json.loads((OUTPUT / "summary.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(OUTPUT / "metrics.csv", index_col="metric")
print("Provider reference data; not station observations or operational backtest evidence.")

In [ ]:
display(
    pd.Series(
        {
            key: summary[key]
            for key in [
                "expected_hours",
                "source_hours",
                "missing_timestamps",
                "duplicate_metric_hours",
                "valid_from_utc",
                "valid_until_utc",
            ]
        }
    )
)
display(metrics[["unit", "present", "missing_pct", "longest_gap_hours"]])
display(pd.read_csv(OUTPUT / "sources.csv"))

In [ ]:
display(Image(filename=str(OUTPUT / "coverage.png"), width=1000))
display(pd.read_csv(OUTPUT / "anomalies.csv"))

IQR flags are review candidates, not invalid values. Rejected corrupt, overlapping or mismatched-city inputs require repair at the source; the report never silently resolves them.

In [ ]:
for warning in summary["warnings"]:
    print("-", warning)